# Approach 1 — Compare two pruning percentages (overlay)

Set `P_1`, `P_2`, and `BATCH_SIZE` in Cell 1.

Produces a **single graph** with both P% curves overlaid:
- Averaged CE scatter (blue = P_1, red = P_2)
- Smooth fitted curve (same colors, solid lines)
- Shared CE_o reference line; separate CE_L and A lines per P%
- BNL vertical marker per P% with IPA annotation

In [7]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
P_1        = 0.98    # first  pruning fraction  (e.g. 0.5 = 50%)
P_2        = 0.96    # second pruning fraction  (e.g. 0.9 = 90%)
BATCH_SIZE = 1024     # single batch size  (64 | 1024 | 60000)
# ─────────────────────────────────────────────────────────────────────────────

import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

BASE_DIR  = r'C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\prune_layers_ALL'
INTER_DIR = r'C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\IPA_methods\Approach_1\intermediate'
OUT_DIR   = r'C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\IPA_methods\Approach_1\test\P_1_v_P_2'

CE_o = np.log(10)   # ln(10) ≈ 2.302585

print(f'P_1={P_1*100:.1f}%   P_2={P_2*100:.1f}%   BS={BATCH_SIZE}   CE_o={CE_o:.6f}')
print('Cell 1 ready.')

P_1=98.0%   P_2=96.0%   BS=1024   CE_o=2.302585
Cell 1 ready.


In [5]:
# === Cell 2 — Load (P_1, BATCH_SIZE) and (P_2, BATCH_SIZE) ===
def load_record(p, bs):
    params_csv = os.path.join(INTER_DIR, f'approach_1_fit_params_bs_{bs}.csv')
    if not os.path.exists(params_csv):
        raise FileNotFoundError(f'Missing params CSV: {params_csv}')
    params_df = pd.read_csv(params_csv)
    params_df.columns = params_df.columns.str.strip()

    row = params_df[np.isclose(params_df['P%'], p * 100)]
    if row.empty:
        raise ValueError(f'No fit params row for P%={p*100:.1f}%  BS={bs}')

    avg_csv = os.path.join(BASE_DIR, f'p-percentage_{p}', f'batch_size_{bs}',
                           f'averaged_runs_p_{p}_bs_{bs}.csv')
    if not os.path.exists(avg_csv):
        raise FileNotFoundError(f'Missing averaged CSV: {avg_csv}')

    avg_df = pd.read_csv(avg_csv)
    avg_df.columns = avg_df.columns.str.strip()
    ce_col = next((c for c in avg_df.columns if c in ('Avg_CE_Test', 'Avg_CE_test')), None)
    bn_col = next((c for c in avg_df.columns if 'Batch' in c), None)
    avg_df = avg_df.dropna(subset=[ce_col, bn_col])

    learn_BN           = float(row['learn_BN'].iloc[0])
    avg_CE_learn_at_BN = float(row['avg_CE_learn_at_BN'].iloc[0]) \
                         if 'avg_CE_learn_at_BN' in row.columns else np.nan
    from_data = np.isfinite(avg_CE_learn_at_BN)

    return {
        'p': p, 'bs': bs,
        'bn_avg':             avg_df[bn_col].values.astype(float),
        'ce_avg':             avg_df[ce_col].values.astype(float),
        'A':                  float(row['A'].iloc[0]),
        'B':                  float(row['B'].iloc[0]),
        'n':                  float(row['n'].iloc[0]),
        'CE_L':               float(row['CE_L'].iloc[0]),
        'learn_BN':           learn_BN,
        'avg_CE_learn_at_BN': avg_CE_learn_at_BN,
        'from_data':          from_data,
        'IPA':                float(row['IPA'].iloc[0]),
    }

rec1 = load_record(P_1, BATCH_SIZE)
rec2 = load_record(P_2, BATCH_SIZE)

for rec in [rec1, rec2]:
    src = 'data' if rec['from_data'] else 'analytic'
    print(f'  P%={rec["p"]*100:5.1f}%  A={rec["A"]:.4f}  CE_L={rec["CE_L"]:.4f}  '
          f'learn_BN={rec["learn_BN"]!r}  IPA={rec["IPA"]}  [{src}]')
print('Cell 2 ready.')

  P%= 88.0%  A=0.3960  CE_L=0.5867  learn_BN=266.0  IPA=0.0064508493955988  [data]
  P%= 90.0%  A=0.5764  CE_L=0.7490  learn_BN=150.0  IPA=0.0103571105579642  [data]
Cell 2 ready.


In [6]:
# === Cell 3 — Overlay both P% curves on a single graph ===
COLORS = ['#1f77b4', '#d62728']   # blue = P_1, red = P_2

plt.rcParams.update({'font.size': 12})
fig, ax = plt.subplots(figsize=(11, 6.5))

# Global x range covering both data sets and both BNL values
bn_end = max(
    rec1['bn_avg'].max(), rec2['bn_avg'].max(),
    rec1['learn_BN'] if np.isfinite(rec1['learn_BN']) else 0,
    rec2['learn_BN'] if np.isfinite(rec2['learn_BN']) else 0,
) * 1.3
bn_smooth = np.linspace(0, bn_end, 800)

for rec, color in zip([rec1, rec2], COLORS):
    p        = rec['p']
    bn_avg   = rec['bn_avg']
    ce_avg   = rec['ce_avg']
    A        = rec['A']
    B        = rec['B']
    n        = rec['n']
    CE_L     = rec['CE_L']
    learn_BN = rec['learn_BN']
    avg_CE_learn_at_BN = rec['avg_CE_learn_at_BN']
    from_data = rec['from_data']
    IPA      = rec['IPA']

    y_fit = A + B / ((bn_smooth + 1) ** n)

    # Averaged CE scatter
    ax.scatter(bn_avg, ce_avg, s=6, color=color, alpha=0.35, zorder=1)

    # Fitted curve
    ax.plot(bn_smooth, y_fit, color=color, linewidth=2.2, zorder=3,
            label=f'P={p*100:.0f}%  A={A:.3f}  B={B:.3f}  n={n:.3f}')

    # CE_L horizontal (colored per P%)
    ax.axhline(CE_L, color=color, linewidth=1.3, linestyle='--', alpha=0.8)
    ax.text(bn_end, CE_L + 0.03,
            f'CE_L={CE_L:.3f}  (P={p*100:.0f}%)',
            ha='right', fontsize=9, color=color)

    # A asymptote (lighter dashed)
    ax.axhline(A, color=color, linewidth=1.0, linestyle=':', alpha=0.6)
    ax.text(bn_end, A - 0.07,
            f'A={A:.3f}  (P={p*100:.0f}%)',
            ha='right', fontsize=9, color=color, alpha=0.8)

    # BNL vertical + annotation
    if np.isfinite(learn_BN) and learn_BN > 0:
        ax.axvline(learn_BN, color=color, linewidth=1.3,
                   linestyle='--' if from_data else ':', alpha=0.8, zorder=4)
        if from_data:
            ax.scatter([learn_BN], [avg_CE_learn_at_BN], s=70,
                       color=color, marker='*', zorder=5)
        src_tag = 'data' if from_data else 'extrap'
        ax.annotate(
            f'BNL={learn_BN:.0f}  P={p*100:.0f}%  [{src_tag}]\nIPA={IPA:.5f}',
            xy=(learn_BN, CE_L),
            xytext=(learn_BN + bn_end * 0.03, CE_L + 0.18),
            fontsize=9, color=color,
            arrowprops=dict(arrowstyle='->', color=color, lw=0.9)
        )

# CE_o — shared, drawn once
ax.axhline(CE_o, color='#888888', linewidth=1.0, linestyle=':')
ax.text(bn_end, CE_o + 0.03, f'CE_o = {CE_o:.4f}',
        ha='right', fontsize=9, color='#666666')

ax.set_xlabel('Batch Number (BN)')
ax.set_ylabel('CE_TEST')
ax.set_xlim(0, bn_end)
ax.set_ylim(max(0, min(rec1['A'], rec2['A']) - 0.15), CE_o + 0.28)
ax.set_title(
    f'Approach 1 — P_1={P_1*100:.1f}% vs P_2={P_2*100:.1f}%  |  BS={BATCH_SIZE}',
    fontsize=13
)
ax.legend(fontsize=10, frameon=False, loc='upper right')
ax.grid(True, alpha=0.25)

out_png = os.path.join(OUT_DIR,
    f'compare_p{int(P_1*100)}_p{int(P_2*100)}_bs{BATCH_SIZE}.png')
fig.tight_layout()
fig.savefig(out_png, dpi=150, bbox_inches='tight')
plt.close(fig)
print(f'Saved: {out_png}')

Saved: C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\IPA_methods\Approach_1\test\P_1_v_P_2\compare_p88_p90_bs1024.png
